# __Modelo Base: Desicion Tree Classifier__

## __Importar librerias__

In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
import matplotlib.pyplot as plt
from imblearn.under_sampling import NearMiss

from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.tree import DecisionTreeClassifier

## __Cargar datos__

In [ ]:
X_cv = pd.read_csv('../out/df/X_cv.csv', index_col=0)
y_cv = pd.read_csv('../out/df/y_cv.csv', index_col=0)

,genero_Masculino,jubilado_Si,en_pareja_Si,dependientes_Si,servicio_telefonico_Si,multiples_lineas_No,multiples_lineas_Si,multiples_lineas_sin servicio teleonico,servicio_internet_DSL,servicio_internet_Fibra_optica,...,tipo_contrato_mes a mes,tipo_contrato_un year,facturacion_electronica_Si,metodo_pago_cheque electronico,metodo_pago_cheque por correo,metodo_pago_tarjeta de credito,metodo_pago_transferencia,meses_contrato,cuenta_mensual,cuentas_total
134,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,3.0,84.30,235.05
4974,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,72.0,24.10,1734.65
6262,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,...,1.0,0.0,1.0,1.0,0.0,0.0,0.0,3.0,49.15,169.05
2718,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,7.0,95.00,655.50
3924,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,20.0,19.25,375.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2206,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,...,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,45.85,45.85
4189,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,69.0,81.50,5553.25
4304,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,23.0,19.65,436.90
1248,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,1.0,7.0,50.30,355.10


## __Analisis de validacion cruzada__

Funcion que calcula e imprime un intervalo de confianza del 95% para una metrica de evaluacion (por defecto, el Recall) obtenida a partir de resultados de validacion cruzada.

Esto ayuda a entender la variabilidad del desempeno del modelo y en que rango se espera que se encuentre la metrica seleccionada.

In [2]:
def cv_score_interval(results, 
                      nombre: str, 
                      score:str='Recall'):
   
    score_mean = np.round(results['test_score'].mean(), 4)
    score_stdev = np.round(results['test_score'].std(), 4)
    lower_b = np.round(score_mean - 2 * score_stdev, 4)
    upper_b = np.round(min(1, score_mean + 2 * score_stdev), 4)
    
    print(f'Recall promedio de {nombre}: {score_mean}')
    print(f'Desviación estándar del Recall de {nombre}: {score_stdev}')
    print(f'El {score} de {nombre} estará entre [{lower_b:.4f},{upper_b:.4f}] con un 95% de confianza')

Aplicar la validacion cruzada al modelo base utilizando los datos _df_

In [8]:
# 6 particiones. Mezcla los datos antes de dividirlos
skf = StratifiedKFold(n_splits=6, shuffle=True, random_state=42)

# instanciar modelo base. Profundidad de 10
baseline_model = DecisionTreeClassifier(max_depth=10, random_state=42)

# Validacion cruzada estratificada. Calcular recall como metrica de desempeno
baseline_cross_val_results = cross_validate(baseline_model, X_cv, y_cv, cv=skf, scoring='recall')

cv_score_interval(
    results=baseline_cross_val_results,
    nombre='Modelo Base',
    score='Recall'
    )

Recall promedio de Modelo Base: 0.5236
Desviación estándar del Recall de Modelo Base: 0.0426
El Recall de Modelo Base estará entre [0.4384,0.6088] con un 95% de confianza


> El modelo base identifica alrededor 52.36% de los casos en los que los clientes cancelan.

> La desviacion estandar es baja, lo que indica que el modelo es estable.

> Con un 95% de confianza, el verdadero recall del modelo al probarlo con nuevos datos estara entre 43.8% y 60.8%. Un rango muy amplio ue se puede mejorar